In [1]:
import pandas as pd

# Load datasets 
phishing_df = pd.read_csv('PhiUSIIL_Phishing_URL_Dataset.csv')
tranco_df = pd.read_csv('tranco_KW6WW.csv')

In [2]:
# Look at each one
print("=== PHISHING DATASET ===")
print(phishing_df.shape)
print(phishing_df.head())
print(phishing_df.columns.tolist())

print("\n=== TRANCO DATASET ===")
print(tranco_df.shape)
print(tranco_df.head())
print(tranco_df.columns.tolist())

=== PHISHING DATASET ===
(235795, 56)
     FILENAME                                 URL  URLLength  \
0  521848.txt    https://www.southbankmosaics.com         31   
1   31372.txt            https://www.uni-mainz.de         23   
2  597387.txt      https://www.voicefmradio.co.uk         29   
3  554095.txt         https://www.sfnmjournal.com         26   
4  151578.txt  https://www.rewildingargentina.org         33   

                       Domain  DomainLength  IsDomainIP  TLD  \
0    www.southbankmosaics.com            24           0  com   
1            www.uni-mainz.de            16           0   de   
2      www.voicefmradio.co.uk            22           0   uk   
3         www.sfnmjournal.com            19           0  com   
4  www.rewildingargentina.org            26           0  org   

   URLSimilarityIndex  CharContinuationRate  TLDLegitimateProb  ...  Pay  \
0               100.0              1.000000           0.522907  ...    0   
1               100.0              0.666

In [3]:
# How many phishing vs legitimate?
print(phishing_df['label'].value_counts())

# Look at just the URL column
print("\n=== PHISHING URLs ===")
print(phishing_df[phishing_df['label']==1]['URL'].head(10).to_string())

print("\n=== LEGITIMATE URLs ===")
print(phishing_df[phishing_df['label']==0]['URL'].head(10).to_string())

label
1    134850
0    100945
Name: count, dtype: int64

=== PHISHING URLs ===
0      https://www.southbankmosaics.com
1              https://www.uni-mainz.de
2        https://www.voicefmradio.co.uk
3           https://www.sfnmjournal.com
4    https://www.rewildingargentina.org
5       https://www.globalreporting.org
6            https://www.saffronart.com
7            https://www.nerdscandy.com
8        https://www.hyderabadonline.in
9                   https://www.aap.org

=== LEGITIMATE URLs ===
11                              http://www.teramill.com
20                          http://www.f0519141.xsph.ru
21                             http://www.shprakserf.gq
27               https://service-mitld.firebaseapp.com/
28                    http://www.kuradox92.lima-city.de
29                          https://liuy-9a930.web.app/
31    https://ipfs.io/ipfs/qmrvvyr84esa2assw9vvwupqj...
32             http://att-103731-107123.weeblysite.com/
34                  https://hidok4f8zl.firebasea

In [4]:
# Flip the labels so 1 = phishing, 0 = legitimate
phishing_df['label'] = phishing_df['label'].map({1: 0, 0: 1})

# Verify the fix
print(phishing_df['label'].value_counts())

print("\n=== PHISHING URLs ===")
print(phishing_df[phishing_df['label']==1]['URL'].head(10).to_string())

print("\n=== LEGITIMATE URLs ===")
print(phishing_df[phishing_df['label']==0]['URL'].head(10).to_string())

label
0    134850
1    100945
Name: count, dtype: int64

=== PHISHING URLs ===
11                              http://www.teramill.com
20                          http://www.f0519141.xsph.ru
21                             http://www.shprakserf.gq
27               https://service-mitld.firebaseapp.com/
28                    http://www.kuradox92.lima-city.de
29                          https://liuy-9a930.web.app/
31    https://ipfs.io/ipfs/qmrvvyr84esa2assw9vvwupqj...
32             http://att-103731-107123.weeblysite.com/
34                  https://hidok4f8zl.firebaseapp.com/
37                                 http://www.ooguy.com

=== LEGITIMATE URLs ===
0      https://www.southbankmosaics.com
1              https://www.uni-mainz.de
2        https://www.voicefmradio.co.uk
3           https://www.sfnmjournal.com
4    https://www.rewildingargentina.org
5       https://www.globalreporting.org
6            https://www.saffronart.com
7            https://www.nerdscandy.com
8        https:/

In [5]:
def is_https(url):
    return 1 if url.startswith('https') else 0

# Test it
print(is_https('https://www.netflix.com'))   # should print 1
print(is_https('http://att-103731.weeblysite.com'))  # should print 0

1
0


In [6]:
def url_length(url):
    return len(url)

print(url_length('https://www.netflix.com'))
print(url_length('http://att-103731.weeblysite.com'))  

23
32


In [7]:
def digit_ratio_in_domain(url):
    from urllib.parse import urlparse
    domain = urlparse(url).netloc  # extracts just the domain part
    domlen = len(domain)
    
    count = 0
    for i in domain:        # loop over the string, not the number
        if i.isdigit():     # check each character
            count += 1      # add 1 every time you find a digit
    
    return count / domlen   # return ratio AFTER the loop

    print(f"domain: {domain}, digits: {count}, length: {domlen}")  # debug line
    return count / domlen
 
print(digit_ratio_in_domain ('http://www.f0519141.xsph.ru'))
print(digit_ratio_in_domain('http://www.netflix.com'))      

0.35
0.0


In [8]:
phishing_df['is_https'] = phishing_df['URL'].apply(is_https)
phishing_df['url_length'] = phishing_df['URL'].apply(url_length)
phishing_df['digit_ratio'] = phishing_df['URL'].apply(digit_ratio_in_domain)

# Now look at the averages by label
print(phishing_df.groupby('label')[['is_https', 'url_length', 'digit_ratio']].mean())

       is_https  url_length  digit_ratio
label                                   
0      1.000000   27.228610     0.003042
1      0.487354   46.238774     0.068511


In [12]:
SUSPICIOUS_TLDS = ['.ru', '.gq', '.ml', '.tk', '.ga', '.cf', '.xyz']

def suspicious_tld(url):
    for tld in SUSPICIOUS_TLDS:
        if url.endswith(tld):  
            return 1
    return 0

print(suspicious_tld('http://www.f0519141.xsph.ru'))
print(suspicious_tld('https://www.netflix.com'))  

1
0


In [13]:
phishing_df['suspicious_tld'] = phishing_df['URL'].apply(suspicious_tld)

print(phishing_df.groupby('label')[['is_https', 'url_length', 'digit_ratio', 'suspicious_tld']].mean())

       is_https  url_length  digit_ratio  suspicious_tld
label                                                   
0      1.000000   27.228610     0.003042        0.006867
1      0.487354   46.238774     0.068511        0.058061
